In [1]:
import argparse
import json
import re
from pathlib import Path
from typing import Any
import os
import pandas as pd

In [2]:
from utils import *

In [3]:
DIR = "/Volumes/DevSSD/Documents/dev/world-cup-sweepstake/"
os.chdir(DIR)

In [4]:
partidos_fase_grupos_path = Path("data/partidos/fase_grupos/partidos_fase_grupos_2022.json")
partidos_fase_final_path = Path("data/partidos/fase_final/partidos_fase_final_2022.json")

In [5]:
path_csv = "data/raw/resultados_2022_all.csv"
ouput_path = "data/raw/resultados_2022_all.json"

In [6]:
df = pd.read_csv(path_csv, sep=";")

In [7]:
df.columns

Index(['Key Id', 'Tournament Id', 'tournament Name', 'Match Id', 'Match Name',
       'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Replayed',
       'Replay', 'Match Date', 'Match Time', 'Stadium Id', 'Stadium Name',
       'City Name', 'Country Name', 'Home Team Id', 'Home Team Name',
       'Home Team Code', 'Away Team Id', 'Away Team Name', 'Away Team Code',
       'Score', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin',
       'Away Team Score Margin', 'Extra Time', 'Penalty Shootout',
       'Score Penalties', 'Home Team Score Penalties',
       'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win',
       'Draw'],
      dtype='str')

In [8]:
colmap = build_column_map(df)

In [9]:
tournament_col = first_existing(colmap, ["Tournament Id", "tournament Name", "Tournament Name"])
stage_col = first_existing(colmap, ["Stage Name"])
group_col = first_existing(colmap, ["Group Name"])

home_team_col = first_existing(colmap, ["Home Team Name"])
away_team_col = first_existing(colmap, ["Away Team Name"])

home_goals_col = first_existing(colmap, ["Home Team Score"])
away_goals_col = first_existing(colmap, ["Away Team Score"])

extra_time_col = first_existing(colmap, ["Extra Time"])
penalty_shootout_col = first_existing(colmap, ["Penalty Shootout"])

result_col = first_existing(colmap, ["Result"])
home_team_win_col = first_existing(colmap, ["Home Team Win"])
away_team_win_col = first_existing(colmap, ["Away Team Win"])
draw_col = first_existing(colmap, ["Draw"])

In [10]:
win_conditions_col = None
for candidate in ["Win conditions", "Win Conditions", "winner_condition", "notes"]:
    normalized = normalize_col_name(candidate)
    if normalized in colmap:
        win_conditions_col = colmap[normalized]
        break

In [11]:
records: list[dict[str, Any]] = []

for _, row in df.iterrows():
    fase = normalize_stage(row[stage_col])
    grupo = normalize_group(row[group_col])

    home_team = translate_team(row[home_team_col])
    away_team = translate_team(row[away_team_col])

    home_goals = int(row[home_goals_col])
    away_goals = int(row[away_goals_col])

    prorroga = csv_bool(row[extra_time_col])
    penaltis = csv_bool(row[penalty_shootout_col])

    item: dict[str, Any] = {
        "fase": fase,
    }

    if fase == "fase_grupos":
        item["grupo"] = grupo

    item.update(
        {
            "equipo_1": home_team,
            "equipo_2": away_team,
            "marcador_equipo_1_90_mins": home_goals,
            "marcador_equipo_2_90_mins": away_goals,
        }
    )

    if fase != "fase_grupos":
        item["prorroga"] = prorroga
        item["penaltis"] = penaltis

    item["ganador"] = infer_winner(
        fase=fase,
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        result=row[result_col],
        home_team_win=row[home_team_win_col],
        away_team_win=row[away_team_win_col],
        draw=row[draw_col],
    )

    records.append(item)

# Add the external_id for the match identifier

In [12]:
partidos_index = load_partidos_index(
    partidos_fase_grupos_path,
    partidos_fase_final_path,
)

In [13]:
records_enriched = []

for record in records:
    key = (
        record["fase"],
        record["equipo_1"],
        record["equipo_2"],
    )

    partido = partidos_index.get(key)

    if partido is None:
        raise ValueError(
            f"No partido found for match: {key}. "
            "Check team names and fase between partidos JSONs and resultados records."
        )

    records_enriched.append({
        "external_id": partido["external_id"],
        "fase": record["fase"],
        "tipo_partido": partido["tipo_partido"],
        "jornada": partido["jornada"],
        "grupo": record.get("grupo"),
        "equipo_1": record["equipo_1"],
        "equipo_2": record["equipo_2"],
        "fecha": partido["fecha"],
        "marcador_equipo_1_90_mins": record["marcador_equipo_1_90_mins"],
        "marcador_equipo_2_90_mins": record["marcador_equipo_2_90_mins"],
        "prorroga": record.get("prorroga", False),
        "penaltis": record.get("penaltis", False),
        "ganador": record["ganador"],
    })

records = records_enriched

# Save

In [14]:
output_path = Path(ouput_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(records)} matches to {output_path}")

Wrote 64 matches to data/raw/resultados_2022_all.json
